In [1]:
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px

pd.options.display.float_format = "{:,.2f}".format
conn = sqlite3.connect("../data/stock_screener.db")

In [ ]:
actions = pd.read_sql_query("SELECT * FROM actions", conn)
fonda = pd.read_sql_query("SELECT * FROM fondamentaux", conn)
prix_long = pd.read_sql_query(
    "SELECT ticker, date, cloture_ajustee FROM prix", conn, parse_dates=["date"]
)

# Une colonne par ticker, une ligne par date avec pivot 
prix = prix_long.pivot(index="date", columns="ticker", values="cloture_ajustee").sort_index()
print(prix.shape)
prix.tail()

(2585, 91)


ticker,AAPL,ABBV,AC.PA,ACA.PA,ADBE,AI.PA,AIR.PA,ALO.PA,AMD,AMZN,...,UPS,URW.PA,V,VIE.PA,VZ,WFC,WMT,XOM,^FCHI,^GSPC
date,,,,,,,,,,,,,,,,,,,,,
2026-10-26,343.91,286.74,45.13,15.10,258.62,175.36,193.40,16.28,531.28,249.09,...,100.32,91.06,404.16,32.44,43.10,82.65,120.29,184.54,"7,847.32","7,676.77"
2026-10-27,347.10,286.39,45.12,15.21,248.36,174.44,192.74,16.18,560.26,253.10,...,106.09,91.86,401.86,32.62,43.93,82.66,119.32,179.91,"7,887.14","7,722.60"
2026-10-28,343.83,285.99,45.99,15.24,249.76,174.24,191.92,15.97,578.44,254.60,...,105.62,91.62,400.60,32.74,43.18,83.67,118.77,182.02,"7,886.96","7,743.98"
2026-10-29,348.91,290.79,46.99,15.30,248.12,177.38,196.34,16.12,566.42,243.79,...,107.33,91.60,410.54,32.85,43.66,84.81,121.25,183.29,"7,970.00","7,720.39"
2026-10-30,344.33,295.81,46.32,14.99,249.76,180.90,192.40,15.81,498.86,239.06,...,108.70,91.04,415.91,33.01,45.04,83.36,123.98,186.51,"7,969.42","7,558.81"


In [ ]:
missing = prix.isna().mean().sort_values(ascending=False)
print("Part of missing values (> 1 %) :")
print(missing[missing > 0.01])

debut = prix.apply(lambda s: s.first_valid_index())
print(debut.sort_values().tail(10))   # 10 latest → eventual exclusion of scorings

Part of missing values (> 1 %) :
ticker
URW.PA   0.65
dtype: float64
ticker
ERF.PA    2016-10-31
ENGI.PA   2016-10-31
EN.PA     2016-10-31
EL.PA     2016-10-31
DSY.PA    2016-10-31
DIS       2016-10-31
DG.PA     2016-10-31
GOOGL     2016-10-31
^GSPC     2016-10-31
URW.PA    2023-04-14
dtype: datetime64[us]


In [ ]:
# goal : coordinate the exchange places
print("NaN avant :", prix.isna().sum().sum())
prix = prix.ffill(limit=5)    # latest known price, up to 5 days
print("NaN après :", prix.isna().sum().sum())

NaN avant : 6237
NaN après : 1668


In [ ]:
#goal : find outliers and identify if they are due to stock splits or other events
rend = prix.pct_change(fill_method=None)

extremes = rend.stack().rename("rendement")
extremes = extremes[extremes.abs() > 0.20].sort_values()
extremes

date        ticker  
2023-10-05  ALO.PA     -0.38
2022-11-10  TEP.PA     -0.34
2026-10-30  INTC       -0.31
2026-04-17  ALO.PA     -0.27
2022-02-03  META       -0.26
2024-08-02  INTC       -0.26
2026-02-06  STLAP.PA   -0.25
2026-07-14  IBM        -0.25
2022-10-27  META       -0.25
2017-05-02  AMD        -0.24
2020-03-16  BA         -0.24
2024-03-07  TEP.PA     -0.23
2020-03-18  SAF.PA     -0.23
2025-04-17  UNH        -0.22
2020-03-18  AIR.PA     -0.22
            CVX        -0.22
2020-03-12  RNO.PA     -0.22
2020-09-08  TSLA       -0.21
2026-02-11  DSY.PA     -0.21
2025-08-01  TEP.PA     -0.21
2024-02-02  META        0.20
2020-03-13  ORCL        0.20
2020-03-26  AIR.PA      0.20
2020-03-24  BA          0.21
            SAF.PA      0.21
2020-11-09  AC.PA       0.22
2024-10-24  TSLA        0.22
2026-08-27  CRM         0.23
2025-04-09  TSLA        0.23
2020-03-24  CVX         0.23
2025-09-18  INTC        0.23
2019-04-16  QCOM        0.23
2023-02-02  META        0.23
2026-04-24  INTC      

In [13]:
# goal : compare the evolution of different tickers, starting from a base of 100
base100 = prix.div(prix.bfill().iloc[0]) * 100
choix = [t for t in ["^GSPC", "^FCHI", "AAPL", "MC.PA", "TTE.PA", "JNJ"] if t in base100.columns]

fig = px.line(base100[choix], title="Évolution base 100")
fig.show()

In [19]:
#goal : compar risk and return ; correlation ? 
stats = pd.DataFrame({
    "rendement_annuel": rend.mean() * 252,   #252 days on average
    "volatilite_annuelle": rend.std() * np.sqrt(252),
})
stats["sharpe_simple"] = stats["rendement_annuel"] / stats["volatilite_annuelle"]
stats = stats.join(actions.set_index("ticker")[["nom", "secteur", "pays", "est_indice"]])
stats["secteur"] = stats["secteur"].fillna("Inconnu")
stats["type"] = np.where(stats["est_indice"] == 1, "indice", "action")

fig = px.scatter(stats, x="volatilite_annuelle", y="rendement_annuel",
                 color="secteur", symbol="type", hover_name="nom",
                 title="Rendement vs risque")
fig.show()
stats.sort_values("sharpe_simple", ascending=False).head(5)

,rendement_annuel,volatilite_annuelle,sharpe_simple,nom,secteur,pays,est_indice,type
ticker,,,,,,,,
NVDA,0.59,0.49,1.18,NVIDIA Corporation,Technology,United States,0,action
LLY,0.35,0.30,1.15,Eli Lilly and Company,Healthcare,United States,0,action
COST,0.23,0.22,1.05,Costco Wholesale Corporation,Consumer Defensive,United States,0,action
AAPL,0.29,0.29,1.02,Apple Inc.,Technology,United States,0,action
AMD,0.57,0.56,1.01,"Advanced Micro Devices, Inc.",Technology,United States,0,action


In [15]:
actions_seules = actions[actions["est_indice"] == 0].sort_values(["secteur", "ticker"])
ordre = [t for t in actions_seules["ticker"] if t in rend.columns]
corr = rend[ordre].corr()

fig = px.imshow(corr, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Corrélation des rendements (triés par secteur)", height=8000)
fig.show()

masque = ~np.eye(len(corr), dtype=bool)
print("Corrélation moyenne entre actions :", round(corr.values[masque].mean(), 2))

Corrélation moyenne entre actions : 0.26


In [22]:
ticker_test = "MC.PA"     # random test ticker
fig = px.histogram(rend[ticker_test].dropna(), nbins=100, title=f"Rendements quotidiens : {ticker_test}")
fig.show()

print("Kurtosis en excès (0 = loi normale) :")
print(rend.kurt().describe())

Kurtosis en excès (0 = loi normale) :
count    91.00
mean     18.01
std      60.88
min       3.87
25%       7.85
50%      10.17
75%      14.14
max     589.10
dtype: float64


In [17]:
f = fonda.merge(actions[["ticker", "secteur", "pays"]], on="ticker")
f[["capitalisation", "per", "rendement_dividende"]].describe()

,capitalisation,per,rendement_dividende
count,89.00,86.00,79.00
mean,"486,788,624,844.22",30.80,0.03
std,"948,094,127,327.01",48.66,0.02
min,0.00,3.11,0.00
25%,"44,197,945,344.00",13.99,0.01
50%,"174,495,498,240.00",22.26,0.02
75%,"395,726,061,568.00",31.06,0.04
max,"5,025,213,972,480.00",434.46,0.11


In [18]:
fig = px.box(f[f["per"] < 100], x="secteur", y="per", title="PER par secteur (PER < 100)")
fig.show()

div = f.groupby("secteur")["rendement_dividende"].mean().sort_values()
fig = px.bar(div, orientation="h", title="Rendement moyen du dividende par secteur")
fig.show()

actions.groupby(["pays", "secteur"], dropna=False).size()

pays           secteur               
France         Basic Materials            1
               Communication Services     2
               Consumer Cyclical          5
               Consumer Defensive         4
               Energy                     1
               Financial Services         4
               Healthcare                 2
               Industrials               10
               Real Estate                1
               Technology                 2
               Utilities                  1
Luxembourg     Basic Materials            1
               Healthcare                 1
Netherlands    Consumer Cyclical          1
               Industrials                1
Switzerland    Technology                 1
United States  Communication Services     5
               Consumer Cyclical          5
               Consumer Defensive         5
               Energy                     2
               Financial Services         7
               Healthcare             

In [ ]:
## Observations
- **Qualité des données** : Seulement 1 acion avec un historique de moins de 3 ans : URW.PA → exclusion dus coring car volatibilité peu fiable.; 
- **Rendement et risque** : Tendance de la volatilité à augmenter avec le risque, rendement les plus elevés s'accompagnent d'une volatibilité élevée (ex : NVidia). S&P500 et CAC40 ont une volatilité similaire mais le S&P500 est presque deux fois plus rentable.
- **Corrélations** : les actions d'un même secteur sont très corrélées.
  Le portefeuille devra limiter le poids d'un secteur. (blocs sectoriels ? France vs États-Unis ?)
- **Distribution des rendements** : (queues épaisses ?)
- **Fondamentaux** : (données manquantes, PER extrêmes)
- **Décisions pour la suite** : (actions à exclure, seuils retenus)


- **Corrélations** : les actions d'un même secteur sont très corrélées.
  Le portefeuille devra limiter le poids d'un secteur.


SyntaxError: invalid character '→' (U+2192) (3637633797.py, line 2)